# Trabalho Individual — AtlasLog
## Análise de dados em Python · Questões A, B, C e E

**Dataset:** `atlaslog_clientes_spss.csv` (uma observação por cliente; variáveis de 12 meses; `churn` = abandono nos 3 meses seguintes).
**Nível de significância:** α = 0,05, salvo indicação contrária.
**Regra:** o CSV original nunca é alterado — todas as transformações são feitas em memória.

> A questão D (segmentação) é feita em KNIME e não consta deste notebook.

<div style="border-left:6px solid #AA8400; background:#FFF4CF; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#AA8400;">COMO USAR ESTE NOTEBOOK</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Cada questão tem: <b>código</b> (produz os números), <b>output</b> e uma célula verde <b>INTERPRETAÇÃO</b> para <b>escrever com as suas palavras</b>, usando os valores do output. O código executado, por si só, não responde à questão.</div>
</div>

## 0. Preparação técnica

In [ ]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.outliers_influence import variance_inflation_factor

ALPHA = 0.05
FICHEIRO = "atlaslog_clientes_spss.csv"
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid")
print("Preparação concluída. α =", ALPHA)

# Questão A — Exploração e descrição dos dados

## A.1 Importação, dimensão, tipos e valores omissos

<div style="border-left:6px solid #2C5F91; background:#EAF2FA; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#2C5F91;">COMO LER ESTE CÓDIGO</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;"><code>pd.read_csv(FICHEIRO, sep=";")</code> lê o CSV (colunas separadas por <code>;</code>). <code>.shape</code> dá (observações, variáveis); <code>.dtypes</code> o tipo de cada coluna; <code>.isna().sum()</code> conta os omissos por coluna.</div>
</div>

In [ ]:
caminho = Path(FICHEIRO)
if not caminho.exists():
    raise FileNotFoundError(f"Coloque {FICHEIRO} na mesma pasta do notebook.")

# Impressão digital do ficheiro: prova que o original não é alterado entre execuções
print("SHA-256 do CSV:", hashlib.sha256(caminho.read_bytes()).hexdigest()[:16], "...")

dados = pd.read_csv(caminho, sep=";")
print("Observações (linhas):", dados.shape[0])
print("Variáveis (colunas):", dados.shape[1])

In [ ]:
tipos = pd.DataFrame({
    "tipo": dados.dtypes.astype(str),
    "omissos": dados.isna().sum(),
    "% omissos": (100 * dados.isna().mean()).round(2),
})
tipos

In [ ]:
print("Linhas duplicadas:", dados.duplicated().sum())
if "cliente_id" in dados.columns:
    print("cliente_id duplicados:", dados["cliente_id"].duplicated().sum())

<div style="border-left:6px solid #3D7054; background:#EAF5EE; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#3D7054;">INTERPRETAÇÃO — A.1 (a escrever)</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Quantos clientes e variáveis? Há tipos de dados inesperados (ex.: número lido como texto)? Que colunas têm omissos, quantos, e que implicação tem para as análises seguintes?</div>
</div>

## A.2 Frequências por tipo de contrato

In [ ]:
freq = dados["tipo_contrato"].value_counts()
tabela_contrato = pd.DataFrame({
    "Frequência": freq,
    "Percentagem (%)": (100 * freq / freq.sum()).round(1),
})
tabela_contrato.loc["Total"] = [freq.sum(), 100.0]
tabela_contrato["Frequência"] = tabela_contrato["Frequência"].astype(int)
print("Omissos em tipo_contrato:", dados["tipo_contrato"].isna().sum())
tabela_contrato

<div style="border-left:6px solid #3D7054; background:#EAF5EE; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#3D7054;">INTERPRETAÇÃO — A.2 (a escrever)</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Qual é o contrato mais frequente (moda) e que peso tem na carteira? A carteira é equilibrada ou concentrada?</div>
</div>

## A.3 Estatísticas descritivas da receita anual

<div style="border-left:6px solid #AA8400; background:#FFF4CF; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#AA8400;">DESVIO-PADRÃO AMOSTRAL</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Usa-se <code>ddof=1</code> (divisor n−1). O <code>.std()</code> do pandas já usa ddof=1 por omissão; o <code>np.std</code> usa ddof=0, e por isso daria um valor diferente.</div>
</div>

In [ ]:
r = dados["receita_anual"].dropna()
desc_receita = pd.Series({
    "n": len(r),
    "Média": r.mean(),
    "Mediana": r.median(),
    "Desvio-padrão amostral (n-1)": r.std(ddof=1),
    "Q1 (25%)": r.quantile(0.25),
    "Q3 (75%)": r.quantile(0.75),
    "Amplitude interquartil (Q3-Q1)": r.quantile(0.75) - r.quantile(0.25),
    "Assimetria": r.skew(),
})
desc_receita.to_frame("receita_anual")

<div style="border-left:6px solid #3D7054; background:#EAF5EE; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#3D7054;">INTERPRETAÇÃO — A.3 (a escrever)</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Compare média e mediana: o que indica sobre a forma? O que diz o desvio-padrão relativamente à média? Entre que valores está o grosso (50%) dos clientes?</div>
</div>

## A.4 Distribuição da receita anual

<div style="border-left:6px solid #2C5F91; background:#EAF2FA; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#2C5F91;">ESCOLHA DO GRÁFICO</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Para uma variável quantitativa contínua, o <b>histograma</b> mostra a forma da distribuição e o <b>boxplot</b> evidencia os valores extremos (regra de Tukey: fora de [Q1 − 1,5·IQR ; Q3 + 1,5·IQR]).</div>
</div>

In [ ]:
q1, q3 = r.quantile(0.25), r.quantile(0.75)
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
extremos = r[(r < lim_inf) | (r > lim_sup)]
print(f"Limites de Tukey: [{lim_inf:,.2f} ; {lim_sup:,.2f}]")
print(f"Valores extremos: {len(extremos)} ({100*len(extremos)/len(r):.1f}%)  |  acima: {(r>lim_sup).sum()}  abaixo: {(r<lim_inf).sum()}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [2, 1]})
sns.histplot(r, bins="auto", kde=True, color="#F2C811", ax=ax[0])
ax[0].axvline(r.mean(), color="C3", ls="--", label=f"Média = {r.mean():,.0f}")
ax[0].axvline(r.median(), color="C0", ls="-", label=f"Mediana = {r.median():,.0f}")
ax[0].set_xlabel("Receita anual (€)")
ax[0].set_ylabel("Número de clientes")
ax[0].set_title("Distribuição da receita anual")
ax[0].legend()
sns.boxplot(y=r, color="#F2C811", ax=ax[1])
ax[1].set_ylabel("Receita anual (€)")
ax[1].set_title("Boxplot (regra de Tukey)")
plt.tight_layout()
plt.show()

<div style="border-left:6px solid #3D7054; background:#EAF5EE; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#3D7054;">INTERPRETAÇÃO — A.4 (a escrever)</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Forma da distribuição (simétrica? assimétrica à direita/esquerda? uni ou multimodal?) apoiada na assimetria, na relação média–mediana e no gráfico. Quantos valores extremos e de que lado? Que cuidado merecem nas análises seguintes?</div>
</div>

# Questão B — Inferência estatística

## B.1 Receita média: Premium versus Standard

<div style="border-left:6px solid #C26D27; background:#FFF0E5; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#C26D27;">PERGUNTA DE GESTÃO</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;"><b>Os clientes Premium geram, em média, mais receita anual do que os Standard?</b></div>
</div>

**a) Hipóteses** (μ_P e μ_S = receita anual média populacional de Premium e Standard):

- H₀: μ_P ≤ μ_S  (Premium não gera mais receita que Standard)
- H₁: μ_P > μ_S  (Premium gera mais receita que Standard) — hipótese direcional, teste **unilateral à direita**

In [ ]:
premium = dados.loc[dados["tipo_contrato"] == "Premium", "receita_anual"].dropna()
standard = dados.loc[dados["tipo_contrato"] == "Standard", "receita_anual"].dropna()

print("Premium : n =", len(premium), "| média =", round(premium.mean(), 2), "| dp =", round(premium.std(ddof=1), 2))
print("Standard: n =", len(standard), "| média =", round(standard.mean(), 2), "| dp =", round(standard.std(ddof=1), 2))

<div style="border-left:6px solid #2C5F91; background:#EAF2FA; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#2C5F91;">COMO LER O TESTE</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;"><code>stats.ttest_ind(premium, standard, equal_var=False, alternative="greater")</code><br><b>equal_var=False</b> → t de Welch (não assume variâncias iguais) · <b>alternative="greater"</b> → H₁: média do 1.º grupo (Premium) &gt; média do 2.º (Standard).</div>
</div>

In [ ]:
welch = stats.ttest_ind(premium, standard, equal_var=False, alternative="greater")

dif = premium.mean() - standard.mean()
print(f"Diferença observada (Premium − Standard): {dif:,.2f} €")
print(f"Diferença relativa à média Standard: {100*dif/standard.mean():.1f}%")
print(f"Estatística t de Welch: {welch.statistic:.4f}")
print(f"Graus de liberdade (Welch–Satterthwaite): {getattr(welch, 'df', float('nan')):.2f}")
print(f"p-value (unilateral, Premium > Standard): {welch.pvalue:.4g}")
ic = welch.confidence_interval(confidence_level=0.95)
print(f"IC 95% unilateral para μ_P − μ_S: [{ic.low:,.2f} ; {ic.high}]")

**Magnitude da diferença** — d de Cohen (diferença padronizada pelo desvio-padrão combinado):

In [ ]:
n1, n2 = len(premium), len(standard)
s_comb = np.sqrt(((n1 - 1) * premium.var(ddof=1) + (n2 - 1) * standard.var(ddof=1)) / (n1 + n2 - 2))
d_cohen = dif / s_comb
print(f"d de Cohen = {d_cohen:.3f}  (referência: ~0,2 pequeno · ~0,5 médio · ~0,8 grande)")

In [ ]:
print(f"Decisão a α = {ALPHA}:",
      "rejeita-se H0 → há evidência de que Premium > Standard"
      if welch.pvalue < ALPHA else
      "não se rejeita H0 → os dados não sustentam que Premium > Standard")

<div style="border-left:6px solid #3D7054; background:#EAF5EE; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#3D7054;">INTERPRETAÇÃO — B.1 (a escrever)</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">c) Diferença em euros e em %, t e p-value; a diferença é pequena/média/grande (d de Cohen) e relevante para o negócio? d) Conclusão a 5%: os dados sustentam a afirmação da Direção? Lembre que significância estatística ≠ relevância prática.</div>
</div>

## B.2 Tipo de contrato e abandono (churn)

<div style="border-left:6px solid #C26D27; background:#FFF0E5; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#C26D27;">PERGUNTA DE GESTÃO</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;"><b>O abandono nos três meses seguintes está associado ao tipo de contrato?</b></div>
</div>

**a) Tabela de contingência** tipo_contrato × churn e percentagem de abandono dentro de cada contrato (percentagens por linha).

In [ ]:
tabela = pd.crosstab(dados["tipo_contrato"], dados["churn"])
print("Valores de churn encontrados:", sorted(dados["churn"].dropna().unique()))
tab_abs = tabela.copy()
tab_abs.columns = [f"churn = {c}" for c in tab_abs.columns]
tab_abs["Total"] = tab_abs.sum(axis=1)
tab_abs

In [ ]:
pct = 100 * pd.crosstab(dados["tipo_contrato"], dados["churn"], normalize="index")
pct.columns = [f"churn = {c} (%)" for c in pct.columns]
pct.round(1)

**b) Hipóteses**

- H₀: tipo_contrato e churn são independentes (a taxa de abandono é igual em todos os contratos)
- H₁: tipo_contrato e churn não são independentes (a taxa de abandono difere entre contratos)

In [ ]:
chi2, p_chi, gl, esperadas_arr = stats.chi2_contingency(tabela)
print(f"Qui-quadrado = {chi2:.4f}")
print(f"Graus de liberdade = {gl}")
print(f"p-value = {p_chi:.4g}")

esperadas = pd.DataFrame(esperadas_arr, index=tabela.index, columns=tabela.columns)
print("\nFrequências esperadas:")
display(esperadas.round(2))
print("Menor frequência esperada:", round(esperadas_arr.min(), 2))
print("Células com E < 5:", int((esperadas_arr < 5).sum()), "de", esperadas_arr.size)
print("Pressuposto (E ≥ 5 em todas/maioria das células):",
      "satisfeito" if (esperadas_arr >= 5).all() else "NÃO totalmente satisfeito — interpretar com cautela")

**Importância prática** — V de Cramér (0 = sem associação; ~0,1 pequena · ~0,3 média · ~0,5 grande, para 2 categorias):

In [ ]:
n_tot = tabela.to_numpy().sum()
v_cramer = np.sqrt(chi2 / (n_tot * min(tabela.shape[0] - 1, tabela.shape[1] - 1)))
print(f"V de Cramér = {v_cramer:.3f}")
print(f"Decisão a α = {ALPHA}:",
      "rejeita-se H0 → há associação entre contrato e churn"
      if p_chi < ALPHA else "não se rejeita H0 → não há evidência de associação")

<div style="border-left:6px solid #3D7054; background:#EAF5EE; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#3D7054;">INTERPRETAÇÃO — B.2 (a escrever)</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">c) Que contrato tem maior/menor taxa de abandono (pontos percentuais de diferença)? Conclusão estatística a 5% (χ², gl, p). V de Cramér: a associação é fraca ou forte? A diferença justifica uma ação comercial? Lembre: churn refere-se aos 3 meses seguintes → associação, não causalidade.</div>
</div>

# Questão C — Correlação e regressão

## C.1 Expedições e receita: dispersão e correlação de Pearson

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=dados, x="expedicoes_mes", y="receita_anual", alpha=0.6)
plt.xlabel("Expedições médias por mês")
plt.ylabel("Receita anual (€)")
plt.title("Receita anual vs expedições mensais")
plt.show()

In [ ]:
par = dados[["expedicoes_mes", "receita_anual"]].dropna()
pearson = stats.pearsonr(par["expedicoes_mes"], par["receita_anual"])
ic_p = pearson.confidence_interval(confidence_level=0.95)
print(f"n = {len(par)}")
print(f"r de Pearson = {pearson.statistic:.4f}")
print(f"p-value = {pearson.pvalue:.3e}")
print(f"IC 95% para ρ = [{ic_p.low:.4f} ; {ic_p.high:.4f}]")
print(f"r² = {pearson.statistic**2:.4f}")

<div style="border-left:6px solid #3D7054; background:#EAF5EE; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#3D7054;">INTERPRETAÇÃO — C.1 (a escrever)</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Direção (sinal de r), intensidade (fraca/moderada/forte), significância (p vs α) e o que o gráfico acrescenta (linearidade, pontos influentes, grupos). Correlação não é causalidade.</div>
</div>

## C.2 Regressão linear simples

In [ ]:
modelo_simples = smf.ols("receita_anual ~ expedicoes_mes", data=dados).fit()
b0, b1 = modelo_simples.params["Intercept"], modelo_simples.params["expedicoes_mes"]
sinal = "+" if b1 >= 0 else "−"
print(f"Equação estimada:  receita_anual^ = {b0:,.2f} {sinal} {abs(b1):,.2f} × expedicoes_mes")
print(f"Declive: p-value = {modelo_simples.pvalues['expedicoes_mes']:.3e} | "
      f"IC 95% = [{modelo_simples.conf_int().loc['expedicoes_mes', 0]:,.2f} ; {modelo_simples.conf_int().loc['expedicoes_mes', 1]:,.2f}]")
print(f"R² = {modelo_simples.rsquared:.4f} | R² ajustado = {modelo_simples.rsquared_adj:.4f}")
print(f"Intervalo observado de expedicoes_mes: [{dados['expedicoes_mes'].min()} ; {dados['expedicoes_mes'].max()}]")

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=dados, x="expedicoes_mes", y="receita_anual", ci=None,
            scatter_kws={"alpha": 0.5}, line_kws={"color": "C3"})
plt.xlabel("Expedições médias por mês")
plt.ylabel("Receita anual (€)")
plt.title("Reta de regressão simples")
plt.show()

<div style="border-left:6px solid #3D7054; background:#EAF5EE; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#3D7054;">INTERPRETAÇÃO — C.2 (a escrever)</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Declive: «por cada expedição adicional por mês, a receita anual esperada varia, em média, em … €». Constante: tem significado prático (expedicoes_mes = 0 está no intervalo observado)? R²: percentagem da variação da receita explicada pelas expedições (e o que NÃO significa: precisão nem causalidade).</div>
</div>

## C.3 Regressão linear múltipla

In [ ]:
FORMULA = ("receita_anual ~ expedicoes_mes + antiguidade_meses + satisfacao + "
           "desconto_pct + entregas_prazo_pct + reclamacoes_12m")
modelo = smf.ols(FORMULA, data=dados).fit()   # a constante é incluída por omissão
print("Observações usadas no modelo:", int(modelo.nobs), "de", len(dados))

ic_m = modelo.conf_int()
coef = pd.DataFrame({
    "coeficiente": modelo.params,
    "erro_padrao": modelo.bse,
    "t": modelo.tvalues,
    "p_value": modelo.pvalues,
    "IC95_inf": ic_m[0],
    "IC95_sup": ic_m[1],
})
coef["significativo (α=0,05)"] = np.where(coef["p_value"] < ALPHA, "sim", "não")
coef

In [ ]:
print(f"R² = {modelo.rsquared:.4f}")
print(f"R² ajustado = {modelo.rsquared_adj:.4f}")
print(f"F({int(modelo.df_model)}; {int(modelo.df_resid)}) = {modelo.fvalue:.3f} | p(F) = {modelo.f_pvalue:.3e}")
print(f"Erro-padrão residual (RMSE do ajuste) = {np.sqrt(modelo.scale):,.2f} €")

Coeficientes **padronizados** (variação em desvios-padrão de Y por 1 desvio-padrão de X) — ajudam a comparar a dimensão relativa, não a «importância» para o negócio:

In [ ]:
preditores = VARS = ["expedicoes_mes", "antiguidade_meses", "satisfacao",
                     "desconto_pct", "entregas_prazo_pct", "reclamacoes_12m"]
dm = dados.loc[modelo.model.data.row_labels]
beta_pad = modelo.params[preditores] * dm[preditores].std() / dm["receita_anual"].std()
beta_pad.to_frame("beta padronizado")

<div style="border-left:6px solid #3D7054; background:#EAF5EE; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#3D7054;">INTERPRETAÇÃO — C.3 (a escrever)</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Teste global (F, p) e R²/R² ajustado. Para as variáveis significativas: sinal, valor e significado de gestão («mantendo as restantes constantes, +1 unidade de X associa-se a … € de receita»). Para as não significativas: não concluir que «não interessam» — apenas que não há evidência parcial neste modelo. Associação ≠ causalidade.</div>
</div>

## C.4 Associação bivariada versus associação condicional

<div style="border-left:6px solid #AA8400; background:#FFF4CF; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#AA8400;">CONTROLAR AS RESTANTES VARIÁVEIS</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">O coeficiente na regressão múltipla mede a associação de X com a receita <b>mantendo as outras variáveis constantes</b> (associação parcial/condicional). Pode diferir muito da correlação bivariada, que mistura o efeito de X com o das variáveis com que X está correlacionado.</div>
</div>

In [ ]:
VAR_C4 = "entregas_prazo_pct"      # <- altere para a variável que escolher
assert VAR_C4 in preditores

par4 = dm[[VAR_C4, "receita_anual"]]
r4 = stats.pearsonr(par4[VAR_C4], par4["receita_anual"])
simples4 = smf.ols(f"receita_anual ~ {VAR_C4}", data=dm).fit()

confronto = pd.DataFrame({
    "Bivariada (simples)": [r4.statistic, simples4.params[VAR_C4], simples4.pvalues[VAR_C4]],
    "Condicional (múltipla)": [np.nan, modelo.params[VAR_C4], modelo.pvalues[VAR_C4]],
}, index=["r de Pearson", "coeficiente (€ por unidade)", "p-value"])
print("Variável escolhida:", VAR_C4)
confronto

**Ilustração de «controlar»** (teorema de Frisch–Waugh–Lovell): retirar de Y e de X a parte explicada pelas outras cinco variáveis e relacionar os resíduos — o declive obtido coincide com o coeficiente da regressão múltipla.

In [ ]:
outras = [v for v in preditores if v != VAR_C4]
res_y = smf.ols("receita_anual ~ " + " + ".join(outras), data=dm).fit().resid
res_x = smf.ols(f"{VAR_C4} ~ " + " + ".join(outras), data=dm).fit().resid
declive_parcial = np.polyfit(res_x, res_y, 1)[0]
print(f"Declive entre resíduos      : {declive_parcial:,.4f}")
print(f"Coeficiente da regressão múltipla: {modelo.params[VAR_C4]:,.4f}")

<div style="border-left:6px solid #3D7054; background:#EAF5EE; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#3D7054;">INTERPRETAÇÃO — C.4 (a escrever)</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Compare r / coeficiente / p bivariados com o coeficiente / p condicional: mudou o sinal, a magnitude ou a significância? Porquê (correlação entre a variável escolhida e as restantes)? Explique por palavras suas o que é «controlar».</div>
</div>

# Questão E — Diagnóstico e análise de sensibilidade

Modelo base: regressão múltipla da questão C.3 (objeto `modelo`).

## E.1 Multicolinearidade: VIF

<div style="border-left:6px solid #AA8400; background:#FFF4CF; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#AA8400;">VIF E TOLERÂNCIA</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">VIF<sub>j</sub> = 1 / (1 − R²<sub>j</sub>), onde R²<sub>j</sub> vem de regredir a variável j nas restantes explicativas. Tolerância = 1/VIF. O erro-padrão do coeficiente aumenta aproximadamente √VIF vezes. Referências habituais: VIF &gt; 5 merece atenção; &gt; 10 sugere problema sério — são <b>heurísticas</b>, não regras absolutas.</div>
</div>

In [ ]:
X = modelo.model.exog
nomes = modelo.model.exog_names
vif = pd.DataFrame({
    "Preditor": nomes[1:],
    "VIF": [variance_inflation_factor(X, i) for i in range(1, X.shape[1])],
})
vif["Tolerância"] = 1 / vif["VIF"]
vif["Inflação do erro-padrão (√VIF)"] = np.sqrt(vif["VIF"])
vif["Sinal de alerta"] = np.where(vif["VIF"] > 10, "VIF > 10", np.where(vif["VIF"] > 5, "VIF > 5", "—"))
vif.set_index("Preditor")

In [ ]:
print("Matriz de correlações entre as explicativas:")
corr_x = dm[preditores].corr()
display(corr_x.round(3))
plt.figure(figsize=(6.5, 5))
sns.heatmap(corr_x, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1)
plt.title("Correlações entre variáveis explicativas")
plt.tight_layout()
plt.show()

<div style="border-left:6px solid #3D7054; background:#EAF5EE; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#3D7054;">INTERPRETAÇÃO — E.1 (a escrever)</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Existe algum VIF elevado? Quais as variáveis envolvidas (veja a matriz de correlações)? Efeitos: erros-padrão inflacionados, p-values menos fiáveis, coeficientes instáveis/sinais inesperados — mas previsão global pouco afetada. Resposta fundamentada <b>sem eliminar automaticamente</b>: por exemplo manter a variável se é relevante para a gestão, combinar/ escolher uma variável por critério de negócio, comparar modelos com e sem ela (sensibilidade), ou recolher mais dados.</div>
</div>

## E.2 Distância de Cook e reestimação sem a observação mais influente

<div style="border-left:6px solid #AA8400; background:#FFF4CF; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#AA8400;">DISTÂNCIA DE COOK</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Mede quanto mudam os coeficientes estimados se uma observação for retirada (combina resíduo grande e alavancagem). Referência habitual: D<sub>i</sub> &gt; 4/n merece análise; D<sub>i</sub> &gt; 1 é muito influente. Uma observação influente <b>não é, por si só, um erro</b> — não se elimina sem justificação.</div>
</div>

In [ ]:
influencia = modelo.get_influence()
cook = pd.Series(influencia.cooks_distance[0], index=modelo.model.data.row_labels)
res_pad = pd.Series(influencia.resid_studentized_internal, index=modelo.model.data.row_labels)
alav = pd.Series(influencia.hat_matrix_diag, index=modelo.model.data.row_labels)

idx_max = cook.idxmax()
print(f"Maior distância de Cook: {cook.max():.4f}  (referência 4/n = {4/len(cook):.4f})")
print("Cliente:", dados.loc[idx_max, "cliente_id"] if "cliente_id" in dados.columns else idx_max)
print(f"Resíduo padronizado: {res_pad[idx_max]:.3f} | alavancagem: {alav[idx_max]:.4f}")
print("\nCaracterísticas do cliente:")
display(dados.loc[[idx_max]].T)

top5 = pd.DataFrame({"cook": cook, "res_padronizado": res_pad, "alavancagem": alav}).nlargest(5, "cook")
if "cliente_id" in dados.columns:
    top5.insert(0, "cliente_id", dados.loc[top5.index, "cliente_id"])
print("Cinco maiores distâncias de Cook:")
top5

In [ ]:
modelo_sem = smf.ols(FORMULA, data=dados.drop(index=idx_max)).fit()

comp = pd.DataFrame({
    "coef (com)": modelo.params,
    "coef (sem)": modelo_sem.params,
})
comp["variação"] = comp["coef (sem)"] - comp["coef (com)"]
comp["variação (%)"] = 100 * comp["variação"] / comp["coef (com)"].abs()
comp["p (com)"] = modelo.pvalues
comp["p (sem)"] = modelo_sem.pvalues
comp["muda significância?"] = np.where((comp["p (com)"] < ALPHA) != (comp["p (sem)"] < ALPHA), "SIM", "não")
comp["muda sinal?"] = np.where(np.sign(comp["coef (com)"]) != np.sign(comp["coef (sem)"]), "SIM", "não")
display(comp.round(4))

ajuste = pd.DataFrame({
    "com": [modelo.rsquared, modelo.rsquared_adj, np.sqrt(modelo.scale), int(modelo.nobs)],
    "sem": [modelo_sem.rsquared, modelo_sem.rsquared_adj, np.sqrt(modelo_sem.scale), int(modelo_sem.nobs)],
}, index=["R²", "R² ajustado", "Erro-padrão residual (€)", "n"])
ajuste

<div style="border-left:6px solid #3D7054; background:#EAF5EE; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#3D7054;">INTERPRETAÇÃO — E.2 (a escrever)</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">Quem é o cliente e porque é influente (resíduo grande? alavancagem alta? valores atípicos?). Que coeficientes mudaram e quanto (%)? Alguma conclusão de gestão mudou (sinal ou significância)? O R² mudou? Conclua sobre a <b>sensibilidade/robustez</b> das conclusões do C.3 a uma única observação e que decisão tomaria sobre a observação (manter, investigar, reportar ambas).</div>
</div>

<div style="border-left:6px solid #943D3D; background:#FBE9E9; padding:12px 16px; border-radius:7px; margin:8px 0 14px 0;">
<div style="font-size:0.82em; font-weight:700; letter-spacing:0.06em; color:#943D3D;">NOTA</div>
<div style="margin-top:6px; font-size:1.05em; color:#2D2D2D;">O enunciado de E.2 que recebi está cortado após «Discuta a…». Se pedir algo adicional (por exemplo, a robustez das conclusões ou outras análises), acrescente-se aqui.</div>
</div>

# Reprodutibilidade

Teste final: reiniciar a sessão, voltar a disponibilizar o CSV e executar todas as células do princípio ao fim, sem erros.

In [ ]:
print("Notebook executado até ao fim.")